# ENM (ProDy) data generation across the extended dataset


In [1]:
import os
import numpy as np
import pandas as pd

from prody import parsePDB, GNM, calcSqFlucts, confProDy

confProDy(verbosity="none")
pd.set_option("display.max_columns", None)

OUT_DIR = "../Output/ENM_dfs"
os.makedirs(OUT_DIR, exist_ok=True)


C:\Users\zcemcel\AppData\Local\anaconda3\envs\Prody\lib\site-packages\prody\utilities\misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
# Load representative structures 
representative_pdb_df = pd.read_csv("../Output/representative_structures.csv")
representative_pdb_df.head(10)


49 representative structures to process across 26 proteins
12 proteins have more than one selected structure (fragmentary/multi-domain coverage)


,uniprot_id,rank,pdb_id,n_residues_this_structure,cumulative_n_residues,n_achievable_residues,cumulative_coverage_frac
0,P06241,1,2DQ7,262,262,431,0.607889
1,P06241,2,1G83,161,423,431,0.981439
2,P07751,1,1U4Q,318,318,382,0.832461
3,P07751,2,1E6H,61,379,382,0.992147
4,P32081,1,1CSP,67,67,67,1.000000
5,P06654,1,1PGX,68,68,124,0.548387
6,P06654,2,3MP9,56,124,124,1.000000
7,P02836,1,2JWT,60,60,60,1.000000
8,P19909,1,2IGG,64,64,220,0.290909
9,P19909,2,2N9K,55,119,220,0.540909


In [13]:
residue_offsets

,uniprot_id,pdb_id,chain,offset,source
0,P06241,1A0N,B,-10,pipeline
1,P05766,1A32,A,1,pipeline
2,P0A6D0,1AOY,A,0,pipeline
3,P07751,1BK2,A,963,pipeline
4,P07751,1E6H,A,963,pipeline
...,...,...,...,...,...
198,D3WAF4,6OBK,A,0,pipeline
199,A0A480YEE4,6SCW,A,974,pipeline
200,P02976,6SOW,A,269,pipeline
201,A9J6U1,6YSE,A,0,pipeline


In [14]:
# Residue numbering offset and chain
residue_offsets = pd.read_csv("../Output/residue_offsets.csv")
print(f"Loaded {len(residue_offsets)} offset entries covering "
      f"{residue_offsets['uniprot_id'].nunique()} proteins")

def resolve_chain_and_offset(uniprot_id, pdb_id, offsets_df):
    matches = offsets_df[(offsets_df["uniprot_id"] == uniprot_id) & (offsets_df["pdb_id"] == pdb_id)]
    if len(matches) >= 1:
        row = matches.iloc[0]
        chain = row["chain"] if pd.notna(row["chain"]) else "A"
        return chain, int(row["offset"]), None
    return "A", None, f"no entry for uniprot_id={uniprot_id}, pdb_id={pdb_id} in residue_offsets.csv"

Loaded 1527 offset entries covering 160 proteins


In [15]:
# GNM fluctuation calculation

def compute_gnm_fluctuations(coords, resnums, max_modes=30, cutoff=10.0):

    n = coords.shape[0]
    n_modes_to_extract = min(max_modes, n - 1)

    gnm = GNM()
    gnm.buildKirchhoff(coords, cutoff=cutoff)
    gnm.calcModes(n - 1)  # 

    fluct_matrix = np.zeros((n, n_modes_to_extract))
    for i in range(1, n_modes_to_extract + 1):
        fluct_matrix[:, i - 1] = calcSqFlucts(gnm[-i])
    cummax = np.maximum.accumulate(fluct_matrix, axis=1)

    df = pd.DataFrame(cummax, columns=[f"Max Fluctuation (-{i})" for i in range(1, n_modes_to_extract + 1)])
    df.insert(0, "Residue", resnums)
    return df


In [21]:
# Fetch, lookup offset, run GNM, save on fluctuation CSV per structure


manifest_rows = []
for row_in in representative_pdb_df.itertuples():
    uniprot_id, pdb_id, rank = row_in.uniprot_id, row_in.pdb_id, row_in.rank
    row = {"uniprot_id": uniprot_id, "pdb_id": pdb_id, "rank": rank, "status": "failed", "reason": ""}
    try:
        chain, offset, reason = resolve_chain_and_offset(uniprot_id, pdb_id, residue_offsets)
        row["chain"] = chain
        if offset is None:
            row["reason"] = reason
            manifest_rows.append(row)
            continue
        structure = parsePDB(pdb_id)
        if structure is None:
            row["reason"] = "could not fetch structure"
            manifest_rows.append(row)
            continue
        calphas = structure.select(f"calpha and chain {chain}".strip())
        if calphas is None or calphas.numAtoms() < 15:
            row["reason"] = "too few CA atoms selected"
            manifest_rows.append(row)
            continue
        pdb_resnums = calphas.getResnums()
        rebased_resnums = pdb_resnums + offset
        out_path = os.path.join(OUT_DIR, f"ENM_{uniprot_id}_{pdb_id}.csv")
        fluct_df = compute_gnm_fluctuations(calphas.getCoords(), rebased_resnums, max_modes=30)
        out_path = os.path.join(OUT_DIR, f"ENM_{uniprot_id}_{pdb_id}.csv")
        fluct_df.to_csv(out_path, index=False)
        row.update({"status": "ok", "reason": "", "offset": offset,
                    "n_residues": len(rebased_resnums), "out_path": out_path})
    except Exception as e:
        row["reason"] = f"exception: {e}"
    manifest_rows.append(row)
enm_manifest = pd.DataFrame(manifest_rows)
enm_manifest.to_csv("enm_manifest.csv", index=False)

enm_manifest

,uniprot_id,pdb_id,rank,status,reason,chain,offset,n_residues,out_path
0,P06241,2DQ7,1,ok,,X,260,263,../Output/ENM_dfs\ENM_P06241_2DQ7.csv
1,P06241,1G83,2,ok,,A,0,161,../Output/ENM_dfs\ENM_P06241_1G83.csv
2,P07751,1U4Q,1,ok,,A,0,318,../Output/ENM_dfs\ENM_P07751_1U4Q.csv
3,P07751,1E6H,2,ok,,A,963,61,../Output/ENM_dfs\ENM_P07751_1E6H.csv
4,P32081,1CSP,1,ok,,A,0,67,../Output/ENM_dfs\ENM_P32081_1CSP.csv
5,P06654,1PGX,1,ok,,A,283,70,../Output/ENM_dfs\ENM_P06654_1PGX.csv
6,P06654,3MP9,2,ok,,A,218,61,../Output/ENM_dfs\ENM_P06654_3MP9.csv
7,P02836,2JWT,1,ok,,A,453,61,../Output/ENM_dfs\ENM_P02836_2JWT.csv
8,P19909,2IGG,1,ok,,A,366,64,../Output/ENM_dfs\ENM_P19909_2IGG.csv
9,P19909,2N9K,2,ok,,A,300,57,../Output/ENM_dfs\ENM_P19909_2N9K.csv


## 5. Combine per-structure fluctuations into one per-protein file

Matches the original single-protein methodology (`VFE-analysis1prody.ipynb`): fast-mode fluctuations from a protein's structures are **pooled by stacking rows**, not by picking a "best" structure or averaging per residue. `pd.concat` across every one of a protein's successfully-processed structures, tagging each row with `PDB_ID`. For PIN1/spg in the original analysis (and by construction of the coverage-based selection above, for the extended dataset too) the selected structures cover largely disjoint residues, so this isn't pseudo-replication -- each row is a genuinely distinct residue.

**Added: percentile-rank normalisation within each structure before pooling.** Raw GNM fluctuation magnitude isn't directly comparable across structures -- the eigenvalue spectrum (and hence the fluctuation scale) depends on network size, and truncated/domain-only constructs show inflated fluctuations near their cut boundary as a known artefact, independent of real local flexibility. Pooling two structures' *raw* fluctuation values and thresholding on the pooled distribution would confound "which domain has the larger overall scale" with "which residues are locally flexible within their own domain." Each `Max Fluctuation (-i)` column gets a companion `Max Fluctuation (-i) [pctile]` column -- that residue's percentile rank (0-1) within its *own* structure's distribution -- so a top-quartile cut on the pooled `[pctile]` columns is calibrated per-structure rather than confounded by cross-structure scale differences. Raw columns are kept alongside (unchanged) for inspection.

In [24]:
# Combine per-structure fluctuations into one per-protein file
# percentile rank transform each Max fluctuation within a structure (0-1 scale)
def add_percentile_normalized_columns(df):

    out = df.copy()
    mode_cols = [c for c in out.columns if c.startswith("Max Fluctuation")]
    for col in mode_cols:
        out[f"{col} [pctile]"] = out[col].rank(pct=True)
    return out


ok_rows = enm_manifest[enm_manifest["status"] == "ok"]

n_combined = 0
for uniprot_id, group in ok_rows.groupby("uniprot_id"):
    per_structure_dfs = []
    for r in group.itertuples():
        df = pd.read_csv(r.out_path)
        df["PDB_ID"] = r.pdb_id
        df = add_percentile_normalized_columns(df)
        per_structure_dfs.append(df)
    combined = pd.concat(per_structure_dfs, ignore_index=True)
    combined.to_csv(os.path.join(OUT_DIR, f"ENM_{uniprot_id}.csv"), index=False)

